In [ ]:
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
cols = ["SD_front", "SD_left", "SD_right", "SD_back", "Class"]
df = pd.read_csv("../wall+following+robot+navigation+data/sensor_readings_4.data", header=None, names=cols)

X = df[cols[:-1]].to_numpy()
classes, y = np.unique(df["Class"], return_inverse=True)

SPLITS = {"20/80": 0.2, "30/70": 0.3, "50/50": 0.5, "70/30": 0.7, "80/20": 0.8}
RODADAS = 100
STATS = ["mean", "std", "max", "min", "median"]

In [ ]:
def treinar(X, y, k):
    return np.array([X[y == c].mean(axis=0) for c in range(k)])  # um centroide por classe


def prever(centroides, X):
    d = ((X[:, None, :] - centroides[None]) ** 2).sum(axis=2)  # distância euclidiana ao quadrado
    return d.argmin(axis=1)


def matriz_confusao(y_true, y_pred, k):
    return np.bincount(y_true * k + y_pred, minlength=k * k).reshape(k, k)

In [ ]:
rng = np.random.default_rng(42)
registros, confusoes = [], {}

for split, frac in SPLITS.items():
    n_tr = int(frac * len(y))
    for r in range(RODADAS):
        idx = rng.permutation(len(y))
        tr, te = idx[:n_tr], idx[n_tr:]

        t0 = time.perf_counter()
        modelo = treinar(X[tr], y[tr], len(classes))
        t1 = time.perf_counter()
        pred = prever(modelo, X[te])
        t2 = time.perf_counter()

        cm = matriz_confusao(y[te], pred, len(classes))
        confusoes[split, r] = cm
        registros.append({
            "split": split, "rodada": r,
            "acuracia": cm.trace() / cm.sum(),
            "t_treino": t1 - t0, "t_teste": t2 - t1,
            **dict(zip(classes, cm.diagonal() / cm.sum(axis=1))),
        })

res = pd.DataFrame(registros)

In [ ]:
grupos = res.groupby("split", sort=False)
tabela = grupos["acuracia"].agg(STATS).join(grupos[["t_treino", "t_teste"]].mean())
display(tabela.round(6))

In [ ]:
plt.figure(figsize=(8, 5))
sns.pointplot(res, x="split", y="acuracia", errorbar="sd")
plt.title("Acurácia (média ± desvio-padrão) por separação treino/teste");

In [ ]:
fig, axes = plt.subplots(len(SPLITS), 2, figsize=(11, 4.5 * len(SPLITS)))
for linha, (split, sub) in zip(axes, res.groupby("split", sort=False)):
    for ax, (rotulo, i) in zip(linha, [("Melhor", sub["acuracia"].idxmax()), ("Pior", sub["acuracia"].idxmin())]):
        cm = confusoes[split, sub.loc[i, "rodada"]]
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=classes, yticklabels=classes, ax=ax)
        ax.set(title=f"{split} | {rotulo} rodada (acc = {sub.loc[i, 'acuracia']:.4f})",
               xlabel="Predito", ylabel="Real")
plt.tight_layout()

In [ ]:
por_classe = res.melt(id_vars="split", value_vars=list(classes),
                      var_name="classe", value_name="acuracia_classe")

for classe, g in por_classe.groupby("classe"):
    print(classe)
    display(g.groupby("split", sort=False)["acuracia_classe"].agg(STATS).round(4))

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(por_classe, x="classe", y="acuracia_classe", hue="split")
plt.title("Acurácia por classe");